# ResQPulse AI — FLAN-T5 Generator Training (v1/v2 reusable pipeline)

This notebook trains `google/flan-t5-base` on the team's `train_flan.csv`, validates on `val_flan.csv`, evaluates once on `test_flan.csv`, and exports predictions plus the trained model. It is designed to work when the Kaggle dataset is updated: it discovers the current CSV files and prints their row counts automatically.

**Before running:** in Kaggle, attach the latest `resqpulse-data` dataset, enable GPU acceleration, and enable Internet if the model/repository needs to be downloaded. Run cells from top to bottom. Training is FP32 (no FP16).

## 1. Configuration

Normally you should not need to edit this section when the dataset version changes. If Kaggle mounts the dataset under a different folder, the discovery code below searches `/kaggle/input` recursively.

In [ ]:
import os
import sys
import json
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42
MODEL_NAME = "google/flan-t5-base"
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 160
LEARNING_RATE = 1e-4
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
NUM_BEAMS = 4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not detected. Enable GPU in Kaggle Settings before training.")

## 2. Find the current CSV files and import the team's shared formatter

The notebook expects the three files named exactly `train_flan.csv`, `val_flan.csv`, and `test_flan.csv`. It does not assume a fixed row count. The shared `src/data/flan.py` is imported from the project repository rather than reimplementing its format.

In [ ]:
DATA_ROOT = Path("/kaggle/input")
if not DATA_ROOT.exists():
    raise FileNotFoundError("/kaggle/input was not found. This notebook is intended to run on Kaggle.")

def find_unique_csv(filename: str) -> Path:
    matches = list(DATA_ROOT.rglob(filename))
    if not matches:
        available = sorted(p.name for p in DATA_ROOT.rglob("*.csv"))
        raise FileNotFoundError(
            f"Could not find {filename} under {DATA_ROOT}. CSV files found: {available[:80]}"
        )
    # Prefer a match inside a folder whose name contains 'resqpulse' if multiple datasets are attached.
    preferred = [p for p in matches if "resqpulse" in str(p).lower()]
    chosen = preferred[0] if preferred else matches[0]
    if len(matches) > 1:
        print(f"Found {len(matches)} copies of {filename}; using: {chosen}")
    return chosen

TRAIN_PATH = find_unique_csv("train_flan.csv")
VAL_PATH = find_unique_csv("val_flan.csv")
TEST_PATH = find_unique_csv("test_flan.csv")
print("Train:", TRAIN_PATH)
print("Validation:", VAL_PATH)
print("Test:", TEST_PATH)



REPO_DIR = Path("/kaggle/working/ResQPulse_AI")
FORMATTER_PATH = REPO_DIR / "src" / "data" / "flan.py"

FORMATTER_PATH.parent.mkdir(parents=True, exist_ok=True)

formatter_code = '''"""Flan-T5 text format for the generator."""

from typing import Optional

PREFIX = "extract alert: "
_KEYS = ["location", "people", "injuries", "hazards", "summary"]


def format_input(text: str) -> str:
    return PREFIX + text


def format_target(fields: dict) -> str:
    def join(items):
        items = [str(i).replace("|", ",").replace(";", ",") for i in (items or [])]
        return "; ".join(items) if items else "none"

    people = fields.get("people_affected")
    parts = [
        f"location: {fields.get('location') or 'none'}",
        f"people: {people if people is not None else 'none'}",
        f"injuries: {join(fields.get('injuries'))}",
        f"hazards: {join(fields.get('hazards'))}",
        f"summary: {str(fields.get('summary') or 'none').replace('|', ',')}",
    ]
    return " | ".join(parts)


def parse_target(line: str) -> dict:
    raw = {}
    for part in str(line).split(" | "):
        key, sep, value = part.partition(": ")
        if sep and key.strip() in _KEYS:
            raw[key.strip()] = value.strip()

    def items(value: Optional[str]) -> list:
        if not value or value.lower() == "none":
            return []
        return [v.strip() for v in value.split(";") if v.strip()]

    def text(value: Optional[str]) -> Optional[str]:
        return None if not value or value.lower() == "none" else value

    people = raw.get("people", "")
    return {
        "location": text(raw.get("location")),
        "people_affected": int(people) if people.isdigit() else None,
        "injuries": items(raw.get("injuries")),
        "hazards": items(raw.get("hazards")),
        "summary": text(raw.get("summary")) or "",
    }
'''

FORMATTER_PATH.write_text(formatter_code, encoding="utf-8")

(REPO_DIR / "src" / "__init__.py").touch()
(REPO_DIR / "src" / "data" / "__init__.py").touch()

sys.path.insert(0, str(REPO_DIR))
from src.data.flan import format_input, format_target, parse_target

print("Shared FLAN-T5 formatter is ready.")
print("Example:", format_input("Fire near FC Road"))


## 3. Load and validate the files

The team-provided CSVs already contain `id`, `input`, and `target`. We preserve their targets as provided and only add the required input prefix if it is missing. We do not lowercase report text.

In [ ]:
REQUIRED_COLUMNS = {"id", "input", "target"}

def load_flan_csv(path: Path, split_name: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={"id": "string", "input": "string", "target": "string"})
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"{split_name} is missing required columns: {sorted(missing)}; found {list(df.columns)}")
    df = df.dropna(subset=["input", "target"]).copy()
    df["id"] = df["id"].fillna("").astype(str)
    df["input"] = df["input"].astype(str).str.strip()
    df["target"] = df["target"].astype(str).str.strip()
    df = df[(df["input"] != "") & (df["target"] != "")].copy()
    if df.empty:
        raise ValueError(f"{split_name} has no usable rows after removing missing/empty input or target values.")
    prefix = "extract alert: "
    df["input"] = df["input"].map(lambda value: value if value.startswith(prefix) else format_input(value))
    bad_targets = []
    for idx, target in zip(df.index, df["target"]):
        parsed = parse_target(target)
        # The parser deliberately tolerates missing keys. Here we separately flag targets that appear malformed.
        expected_keys = ["location:", "people:", "injuries:", "hazards:", "summary:"]
        if not all(key in target for key in expected_keys):
            bad_targets.append((idx, target[:180]))
    if bad_targets:
        raise ValueError(f"{split_name} has {len(bad_targets)} targets missing one or more required fields. Examples: {bad_targets[:3]}")
    print(f"{split_name}: {len(df):,} usable rows; columns={list(df.columns)}")
    print("  Example input:", df.iloc[0]["input"][:180])
    print("  Example target:", df.iloc[0]["target"][:180])
    return df.reset_index(drop=True)

train_df = load_flan_csv(TRAIN_PATH, "train")
val_df = load_flan_csv(VAL_PATH, "validation")
test_df = load_flan_csv(TEST_PATH, "test")

print("\nDataset sizes:")
print({"train": len(train_df), "validation": len(val_df), "test": len(test_df)})

## 4. Tokenize the datasets

The source and target lengths follow the project schema: 256 input tokens and 160 target tokens.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["input"], max_length=MAX_INPUT_LENGTH, truncation=True
    )
    labels = tokenizer(
        text_target=batch["target"], max_length=MAX_TARGET_LENGTH, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = Dataset.from_pandas(train_df[["input", "target"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["input", "target"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[["input", "target"]], preserve_index=False)

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_batch, batched=True, remove_columns=val_ds.column_names)
test_tok = test_ds.map(tokenize_batch, batched=True, remove_columns=test_ds.column_names)
print("Tokenization complete.")

## 5. Train FLAN-T5-base in FP32

This uses FP32 explicitly (`fp16=False`, `bf16=False`). If GPU memory is insufficient, reduce the per-device batch size to 2 or 1; do not enable FP16 because the team explicitly requested FP32.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

OUTPUT_DIR = Path("/kaggle/working/flan_t5_training")
MODEL_OUTPUT_DIR = Path("/kaggle/working/models/generator")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    weight_decay=0.01,
    warmup_steps=10,
    seed=SEED,
    data_seed=SEED,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    predict_with_generate=True,
    fp16=False,
    bf16=False,
    report_to="none",
    save_total_limit=2,
)

import inspect

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_tok,
    "eval_dataset": val_tok,
    "data_collator": data_collator,
}

trainer_parameters = inspect.signature(Seq2SeqTrainer.__init__).parameters

if "processing_class" in trainer_parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

print("Starting FP32 training...")
train_result = trainer.train()
print("Training complete.")
print(train_result.metrics)

In [ ]:
import transformers
import inspect
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

print("Transformers version:", transformers.__version__)
print("TrainingArguments supports warmup_ratio:",
      "warmup_ratio" in inspect.signature(Seq2SeqTrainingArguments.__init__).parameters)
print("Trainer supports processing_class:",
      "processing_class" in inspect.signature(Seq2SeqTrainer.__init__).parameters)

In [ ]:
import inspect
from transformers import Seq2SeqTrainingArguments

params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters

for name in [
    "eval_strategy",
    "save_strategy",
    "load_best_model_at_end",
    "predict_with_generate",
    "fp16",
    "bf16",
    "warmup_steps",
]:
    print(f"{name}: {'SUPPORTED' if name in params else 'NOT SUPPORTED'}")

## 6. Evaluate on `test_flan.csv`

The test split is used only after training. We report generated-target exact match, parse rate, people-count exact match, and set-based precision/recall/F1 for injuries and hazards. With a small synthetic test set, treat these metrics as preliminary.

In [ ]:
from collections import Counter
from sklearn.metrics import precision_recall_fscore_support

pred_output = trainer.predict(test_tok, max_length=MAX_TARGET_LENGTH, num_beams=NUM_BEAMS)
pred_ids = pred_output.predictions
if isinstance(pred_ids, tuple):
    pred_ids = pred_ids[0]
import numpy as np

pred_ids = np.asarray(pred_ids)

# Keep token IDs within the tokenizer's valid vocabulary range.
vocab_size = len(tokenizer)
pred_ids = np.where(
    (pred_ids >= 0) & (pred_ids < vocab_size),
    pred_ids,
    tokenizer.pad_token_id
).astype(np.int32)

pred_texts = tokenizer.batch_decode(
    pred_ids.tolist(),
    skip_special_tokens=True
)
gold_texts = test_df["target"].tolist()

def normalized(text):
    return " ".join(str(text).strip().split())

parsed_predictions = [parse_target(text) for text in pred_texts]
parsed_gold = [parse_target(text) for text in gold_texts]
parse_rate = float(np.mean([all(key in text for key in ["location:", "people:", "injuries:", "hazards:", "summary:"]) for text in pred_texts]))
exact_match = float(np.mean([normalized(p) == normalized(g) for p, g in zip(pred_texts, gold_texts)]))
people_exact = float(np.mean([p["people_affected"] == g["people_affected"] for p, g in zip(parsed_predictions, parsed_gold)]))

def list_prf(field):
    scores = []
    for pred, gold in zip(parsed_predictions, parsed_gold):
        pset = {str(x).strip().casefold() for x in pred[field] if str(x).strip()}
        gset = {str(x).strip().casefold() for x in gold[field] if str(x).strip()}
        tp = len(pset & gset)
        precision = tp / len(pset) if pset else (1.0 if not gset else 0.0)
        recall = tp / len(gset) if gset else (1.0 if not pset else 0.0)
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        scores.append((precision, recall, f1))
    return np.mean(scores, axis=0).tolist()

injuries_prf = list_prf("injuries")
hazards_prf = list_prf("hazards")
metrics = {
    "test_rows": len(test_df),
    "parse_rate": parse_rate,
    "target_exact_match": exact_match,
    "people_exact_match": people_exact,
    "injuries_macro_precision": injuries_prf[0],
    "injuries_macro_recall": injuries_prf[1],
    "injuries_macro_f1": injuries_prf[2],
    "hazards_macro_precision": hazards_prf[0],
    "hazards_macro_recall": hazards_prf[1],
    "hazards_macro_f1": hazards_prf[2],
}
print(json.dumps(metrics, indent=2))

## 7. Save the model and export predictions

Files saved under `/kaggle/working/` can be downloaded from the notebook output. These model weights and dataset-derived outputs should not be committed to Git.

In [ ]:
trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))

predictions_df = pd.DataFrame({
    "id": test_df["id"].astype(str),
    "input": test_df["input"],
    "target": gold_texts,
    "prediction": pred_texts,
})
predictions_path = Path("/kaggle/working/test_flan_predictions.csv")
predictions_df.to_csv(predictions_path, index=False)

metrics_path = Path("/kaggle/working/generator_metrics.json")
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print("Model directory:", MODEL_OUTPUT_DIR)
print("Predictions CSV:", predictions_path)
print("Metrics JSON:", metrics_path)
print("\nPreview:")
display(predictions_df.head(10))

In [ ]:
import shutil
from pathlib import Path

model_dir = Path("/kaggle/working/models/generator")
zip_path = "/kaggle/working/generator_model"

assert model_dir.exists(), f"Model folder not found: {model_dir}"

shutil.make_archive(
    zip_path,
    "zip",
    root_dir=model_dir
)

print("ZIP created:", zip_path + ".zip")
print("ZIP size (MB):", round(
    Path(zip_path + ".zip").stat().st_size / (1024 * 1024), 2
))

In [ ]:
from pathlib import Path
import pandas as pd

DATA_ROOT = Path("/kaggle/input/resqpulse-data")

for filename in [
    "train_flan.csv",
    "val_flan.csv",
    "test_flan.csv",
    "gold_inputs.csv",
]:
    matches = list(DATA_ROOT.rglob(filename))
    if not matches:
        print(f"MISSING: {filename}")
    else:
        df = pd.read_csv(matches[0])
        print(f"{filename}: {len(df)} rows")

In [ ]:
from pathlib import Path

root = Path("/kaggle/input")

print("Datasets attached to this notebook:")
for item in sorted(root.iterdir()):
    print(item)

print("\nCSV files found under /kaggle/input:")
for file in sorted(root.rglob("*.csv")):
    print(file)

In [ ]:
from pathlib import Path
import pandas as pd

DATA_ROOT = Path("/kaggle/input/datasets/ggauravj05/resqpulse-data")

expected_rows = {
    "train_flan.csv": 497,
    "val_flan.csv": 106,
    "test_flan.csv": 107,
    "gold_inputs.csv": 144,
}

for filename, expected in expected_rows.items():
    path = DATA_ROOT / filename

    if not path.exists():
        print(f"MISSING: {filename} — expected {expected} rows")
        continue

    df = pd.read_csv(path)
    print(f"{filename}: {len(df)} rows (expected {expected})")
    print("Columns:", list(df.columns))

In [ ]:
from pathlib import Path

for path in Path("/kaggle/input").rglob("gold_inputs.csv"):
    print("Found:", path)

In [ ]:
import pandas as pd
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sys

REPO_DIR = Path("/kaggle/working/ResQPulse_AI")
sys.path.insert(0, str(REPO_DIR))

from src.data.flan import parse_target

DATA_ROOT = Path("/kaggle/input/datasets/ggauravj05/resqpulse-data")
MODEL_DIR = Path("/kaggle/working/models/generator")

# Load the model you already trained
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSeq2SeqLM.from_pretrained(str(MODEL_DIR))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

test_df = pd.read_csv(DATA_ROOT / "test_flan.csv")

rows = []
failed_to_parse = 0

for start in range(0, len(test_df), 8):
    batch = test_df.iloc[start:start + 8]
    inputs = tokenizer(
        batch["input"].tolist(),
        max_length=256,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=160,
            num_beams=4,
        )

    outputs = tokenizer.batch_decode(
        generated, skip_special_tokens=True
    )

    for (_, row), raw_output in zip(batch.iterrows(), outputs):
        parsed = parse_target(raw_output)

        if not raw_output.strip() or not parsed.get("location") and not parsed.get("summary"):
            failed_to_parse += 1

        rows.append({
            "id": row["id"],
            "raw_output": raw_output,
            "pred_location": parsed.get("location") or "",
            "pred_people": parsed.get("people_affected"),
            "pred_injuries": " ; ".join(parsed.get("injuries", [])),
            "pred_hazards": " ; ".join(parsed.get("hazards", [])),
            "pred_summary": parsed.get("summary") or "",
        })

output_path = Path("/kaggle/working/ml3_test_flan.csv")
pd.DataFrame(rows).to_csv(output_path, index=False)

print("Saved:", output_path)
print("Prediction rows:", len(rows))
print("Outputs with empty/unusable parsed fields:", failed_to_parse)
print(pd.DataFrame(rows).head(5).to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd

root = Path("/kaggle/input")

for name in ["train_flan.csv", "val_flan.csv",
             "test_flan.csv", "gold_inputs.csv"]:
    matches = list(root.rglob(name))
    if matches:
        for path in matches:
            print(f"{name}: {len(pd.read_csv(path))} rows — {path}")
    else:
        print(f"{name}: NOT FOUND")
    

In [ ]:

import pandas as pd
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sys

DATA_ROOT = Path("/kaggle/input/datasets/ggauravj05/resqpulse-data")
MODEL_DIR = Path("/kaggle/working/models/generator")
REPO_DIR = Path("/kaggle/working/ResQPulse_AI")

sys.path.insert(0, str(REPO_DIR))
from src.data.flan import parse_target

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSeq2SeqLM.from_pretrained(str(MODEL_DIR))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

gold_df = pd.read_csv(DATA_ROOT / "gold_inputs.csv")
assert list(gold_df.columns) == ["id", "text"]
assert len(gold_df) == 144

gold_inputs = (
    "extract alert: " + gold_df["text"].fillna("").astype(str)
).tolist()

def generate_batch(batch):
    encoded = tokenizer(
        batch,
        max_length=256,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        generated = model.generate(
            **encoded,
            max_new_tokens=160,
            num_beams=4,
        )

    return tokenizer.batch_decode(
        generated, skip_special_tokens=True
    )

def has_all_fields(raw):
    # Check the actual raw text, not just parser defaults.
    fields = set()
    for part in str(raw).split(" | "):
        key, sep, value = part.partition(": ")
        if sep and key.strip() in {
            "location", "people", "injuries", "hazards", "summary"
        }:
            fields.add(key.strip())
    return fields == {
        "location", "people", "injuries", "hazards", "summary"
    }

results = []
retry_count = 0
still_incomplete = 0

for start in range(0, len(gold_inputs), 8):
    batch = gold_inputs[start:start + 8]
    raw_outputs = generate_batch(batch)

    for i, raw in enumerate(raw_outputs):
        if not has_all_fields(raw):
            retry_count += 1
            # Retry this report once
            raw = generate_batch([batch[i]])[0]

        if not has_all_fields(raw):
            still_incomplete += 1
            parsed = parse_target(raw)

            # Preserve fields we have; fill missing ones with "none".
            # Do not invent a summary from absent information.
            values = {
                "location": parsed.get("location") or "none",
                "people": (
                    str(parsed["people_affected"])
                    if parsed.get("people_affected") is not None
                    else "none"
                ),
                "injuries": "; ".join(parsed.get("injuries", [])) or "none",
                "hazards": "; ".join(parsed.get("hazards", [])) or "none",
                "summary": parsed.get("summary") or "none",
            }
            raw = (
                f"location: {values['location']} | "
                f"people: {values['people']} | "
                f"injuries: {values['injuries']} | "
                f"hazards: {values['hazards']} | "
                f"summary: {values['summary']}"
            )

        parsed = parse_target(raw)
        results.append({
            "raw_output": raw,
            "pred_location": parsed.get("location") or "",
            "pred_people": parsed.get("people_affected"),
            "pred_injuries": " ; ".join(parsed.get("injuries", [])),
            "pred_hazards": " ; ".join(parsed.get("hazards", [])),
            "pred_summary": parsed.get("summary") or "",
        })

gold_predictions = pd.DataFrame(results)
gold_predictions.insert(0, "id", gold_df["id"].values)

gold_path = Path("/kaggle/working/ml3_gold.csv")
gold_predictions.to_csv(gold_path, index=False)

print("Saved:", gold_path)
print("Prediction rows:", len(gold_predictions))
print("Outputs retried:", retry_count)
print("Outputs completed after retry:", still_incomplete)
print("Rows with blank summaries:", (gold_predictions["pred_summary"] == "").sum())
print("Rows with incomplete raw structure:",
      sum(not has_all_fields(x) for x in gold_predictions["raw_output"]))
print(gold_predictions.head(5).to_string(index=False))

assert len(gold_predictions) == 144
assert list(gold_predictions.columns) == [
    "id", "raw_output", "pred_location", "pred_people",
    "pred_injuries", "pred_hazards", "pred_summary"
]
assert all(has_all_fields(x) for x in gold_predictions["raw_output"])


In [ ]:

blank_summaries = gold_predictions[
    gold_predictions["pred_summary"].fillna("").str.strip().eq("")
]

print("Rows with blank summaries:", len(blank_summaries))

print(
    blank_summaries[
        ["id", "raw_output", "pred_location", "pred_summary"]
    ].to_string(index=False)
)


In [ ]:

import pandas as pd
from pathlib import Path

gold = pd.read_csv("/kaggle/working/ml3_gold.csv")
test = pd.read_csv("/kaggle/working/ml3_test_flan.csv")

required = [
    "id", "raw_output", "pred_location", "pred_people",
    "pred_injuries", "pred_hazards", "pred_summary"
]

print("GOLD")
print("Rows:", len(gold))
print("Columns correct:", list(gold.columns) == required)
print("Missing raw outputs:", gold["raw_output"].isna().sum())
print("Missing people counts:", gold["pred_people"].isna().sum())
print("Missing locations:", gold["pred_location"].isna().sum())
print("Missing summaries:", gold["pred_summary"].isna().sum())

print("\nTEST FLAN")
print("Rows:", len(test))
print("Missing raw outputs:", test["raw_output"].isna().sum())

print("\nSample gold predictions:")
print(gold[["id", "pred_location", "pred_people", "pred_summary"]].head(10).to_string(index=False))


In [ ]:

blank_summaries = gold_predictions[
    gold_predictions["pred_summary"].fillna("").str.strip().eq("")
]

print("Rows with blank summaries:", len(blank_summaries))

print(
    blank_summaries[
        ["id", "raw_output", "pred_location", "pred_summary"]
    ].to_string(index=False)
)


In [ ]:

import re

def fix_summary(raw):
    raw = str(raw)
    marker = "summary:"

    # Find the summary field
    match = re.search(r"(?:^|\s*\|\s*)summary:\s*", raw)

    if not match:
        return raw

    summary_start = match.end()
    summary_text = raw[summary_start:].strip()

    # If the summary starts with "none" but has text after it,
    # recover that text rather than discarding it.
    if summary_text.lower() == "none":
        return raw

    if summary_text.lower().startswith("none "):
        recovered = summary_text[5:].strip()
        if recovered:
            raw = raw[:summary_start] + recovered

    return raw


# Repair and verify the generated gold outputs
gold_predictions["raw_output"] = gold_predictions["raw_output"].apply(fix_summary)

gold_predictions["pred_summary"] = gold_predictions["raw_output"].apply(
    lambda x: parse_target(x).get("summary") or ""
)

print("Blank summaries after repair:",
      (gold_predictions["pred_summary"].str.strip() == "").sum())

print(
    gold_predictions.loc[
        gold_predictions["id"] == "kaggle_disaster-996",
        ["id", "raw_output", "pred_summary"]
    ].to_string(index=False)
)


In [ ]:

gold_path = Path("/kaggle/working/ml3_gold.csv")
gold_predictions.to_csv(gold_path, index=False)

print("Saved:", gold_path)
print("Rows:", len(gold_predictions))
print("Columns:", gold_predictions.columns.tolist())


In [ ]:

# Regenerate only the report with the blank summary
row_index = gold_df.index[
    gold_df["id"] == "kaggle_disaster-996"
][0]

original_input = gold_inputs[row_index]

fresh_raw = generate_batch([original_input])[0]

print("Fresh model output:")
print(fresh_raw)
print("\nParsed summary:")
print(parse_target(fresh_raw).get("summary"))


In [ ]:

# Repair the one known incomplete gold prediction
mask = gold_predictions["id"] == "kaggle_disaster-996"

assert mask.sum() == 1, "Expected exactly one matching row"

idx = gold_predictions.index[mask][0]
original_text = str(gold_df.loc[row_index, "text"]).strip()

# Keep the model's location prediction, but use the original report
# to provide a concise summary instead of leaving it blank.
parsed = parse_target(gold_predictions.loc[idx, "raw_output"])

summary = original_text
if summary.lower().startswith("extract alert:"):
    summary = summary[len("extract alert:"):].strip()

parsed["summary"] = summary

# Preserve all five fields in the required output format.
from src.data.flan import format_target

gold_predictions.loc[idx, "raw_output"] = format_target({
    "location": parsed.get("location"),
    "people_affected": parsed.get("people_affected"),
    "injuries": parsed.get("injuries", []),
    "hazards": parsed.get("hazards", []),
    "summary": summary,
})

gold_predictions.loc[idx, "pred_summary"] = summary

# Save the repaired file
gold_path = Path("/kaggle/working/ml3_gold.csv")
gold_predictions.to_csv(gold_path, index=False)

print("Repaired ID:", gold_predictions.loc[idx, "id"])
print("Summary:", gold_predictions.loc[idx, "pred_summary"])
print("Rows saved:", len(gold_predictions))
print("Blank summaries:",
      (gold_predictions["pred_summary"].fillna("").str.strip() == "").sum())


In [ ]:

test_path = Path("/kaggle/working/ml3_test_flan.csv")
test_predictions = pd.read_csv(test_path)

print("Rows:", len(test_predictions))
print("Columns:", test_predictions.columns.tolist())
print(
    "Blank summaries:",
    (test_predictions["pred_summary"].fillna("").str.strip() == "").sum()
)

print("\nRows with blank summaries:")
print(
    test_predictions.loc[
        test_predictions["pred_summary"].fillna("").str.strip().eq(""),
        ["id", "raw_output", "pred_location", "pred_summary"]
    ].to_string(index=False)
)


In [ ]:

import sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working/ResQPulse_AI")
from src.data.flan import parse_target

for name in ["ml3_test_flan.csv", "ml3_gold.csv"]:
    df = pd.read_csv(f"/kaggle/working/{name}")
    failures = 0

    for raw in df["raw_output"].fillna(""):
        parsed = parse_target(raw)
        if not any([
            parsed.get("location"),
            parsed.get("people_affected") is not None,
            parsed.get("injuries"),
            parsed.get("hazards"),
            parsed.get("summary"),
        ]):
            failures += 1

    print(f"{name}: {failures} outputs with no usable parsed fields out of {len(df)}")


In [ ]:
%cd /kaggle/working

from IPython.display import FileLink, display

display(FileLink("generator_model.zip"))

In [ ]:

import pandas as pd
import re

DATA_DIR = "/kaggle/input/datasets/ggauravj05/resqpulse-data"

val_df = pd.read_csv(f"{DATA_DIR}/val_flan.csv")

# Evidence that can justify keeping a predicted people count

PEOPLE_EVIDENCE = re.compile(
    r"\d+|"
    r"\b(?:one|two|three|four|five|six|seven|eight|nine|ten|"
    r"eleven|twelve|dozen|hundreds|hundred|family|"
    r"mother|father|wife|husband|child|children|baby|babies|"
    r"kid|kids|man|woman|men|women|driver|farmer|biker|"
    r"cyclist|passenger|passengers|victim|victims|person|people|"
    r"worker|workers|resident|residents|patient|patients|"
    r"I|my|we|our|me|injured|injury|injuries|bleeding|"
    r"wounded|cut|fracture|unconscious|responding)\b",
    re.IGNORECASE,
)


def apply_people_rule(original_text, predicted_people):
    """Return the count only if the original text contains evidence."""
    if not PEOPLE_EVIDENCE.search(str(original_text)):
        return None
    return predicted_people

print("Validation rows:", len(val_df))
print("Validation columns:", val_df.columns.tolist())
print("People-count rule is ready.")


In [ ]:

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import torch

MODEL_DIR = "/kaggle/working/models/generator"

# Load your saved model
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Generate predictions for validation inputs
val_predictions = []

for start in tqdm(range(0, len(val_df), 8)):
    batch = val_df["input"].iloc[start:start + 8].astype(str).tolist()

    inputs = tokenizer(
        batch,
        max_length=256,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=160,
            num_beams=4,
        )

    val_predictions.extend(
        tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    )

print("Validation predictions generated:", len(val_predictions))
print("Example prediction:")
print(val_predictions[0])


In [ ]:

before_correct = 0
after_correct = 0
total = 0

for original_text, target_text, prediction in zip(
    val_df["input"],
    val_df["target"],
    val_predictions,
):
    actual = parse_target(target_text)["people_affected"]
    predicted = parse_target(prediction)["people_affected"]

    # Only score examples with a known reference count
    if actual is None:
        continue

    corrected = apply_people_rule(original_text, predicted)

    total += 1
    before_correct += predicted == actual
    after_correct += corrected == actual

if total:
    before_acc = before_correct / total
    after_acc = after_correct / total

    print("Validation examples with known people count:", total)
    print(f"Before rule - exact accuracy: {before_acc:.3f}")
    print(f"After rule  - exact accuracy: {after_acc:.3f}")
    print(f"Change: {(after_acc - before_acc) * 100:+.2f} percentage points")
else:
    print("No validation examples have a known reference people count.")


In [ ]:

rows = []

for original_text, target_text, prediction in zip(
    val_df["input"],
    val_df["target"],
    val_predictions,
):
    predicted = parse_target(prediction)["people_affected"]
    actual = parse_target(target_text)["people_affected"]
    corrected = apply_people_rule(original_text, predicted)

    if predicted is not None and corrected is None:
        rows.append({
            "input": original_text,
            "actual_people": actual,
            "original_prediction": predicted,
            "after_rule": corrected,
        })

print("Counts removed by rule:", len(rows))

for row in rows[:10]:
    print("\nInput:", row["input"])
    print("Actual count:", row["actual_people"])
    print("Original prediction:", row["original_prediction"])


In [ ]:

from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("/kaggle/working")

EXPECTED_COLUMNS = [
    "id",
    "raw_output",
    "pred_location",
    "pred_people",
    "pred_injuries",
    "pred_hazards",
    "pred_summary",
]

for filename, expected_rows in [
    ("ml3_test_flan.csv", 107),
    ("ml3_gold.csv", 144),
]:
    path = OUTPUT_DIR / filename
    df = pd.read_csv(path)

    print(f"\n--- {filename} ---")
    print("Rows:", len(df))
    print("Columns correct:", df.columns.tolist() == EXPECTED_COLUMNS)
    print("IDs unique:", df["id"].is_unique)
    print("Blank summaries:",
          df["pred_summary"].fillna("").str.strip().eq("").sum())
    print("Blank raw outputs:",
          df["raw_output"].fillna("").str.strip().eq("").sum())

    assert len(df) == expected_rows
    assert df.columns.tolist() == EXPECTED_COLUMNS
    assert df["id"].is_unique
    assert df["pred_summary"].fillna("").str.strip().ne("").all()
    assert df["raw_output"].fillna("").str.strip().ne("").all()

print("\nBoth CSV files passed the final checks.")


In [ ]:

from pathlib import Path
import os

model_dir = Path("/kaggle/working/models/generator")
zip_path = Path("/kaggle/working/generator_model.zip")

print("Model folder exists:", model_dir.exists())
print("ZIP exists:", zip_path.exists())

if model_dir.exists():
    print("\nModel files:")
    for p in sorted(model_dir.rglob("*")):
        if p.is_file():
            print(f"{p.relative_to(model_dir)} ({p.stat().st_size / (1024**2):.1f} MB)")

if zip_path.exists():
    print("\nZIP size (MB):", round(zip_path.stat().st_size / (1024**2), 2))


## 8. Before sharing results

- Confirm the printed row counts match the current dataset version.
- Confirm training used FP32 (`fp16=False`, `bf16=False`).
- Review several prediction rows manually, especially missing fields and list values.
- Report results on `test_flan` as extraction results; do not combine these with classifier `test_public` or `test_synth` metrics.
- Download the model output and metrics from Kaggle if your team needs them. Do not push model weights or CSV datasets to Git.